In [ ]:
dbutils.widgets.text("catalog", "hls_glucosphere", "Catalog")
dbutils.widgets.text("schema", "cgm", "Schema")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")
print(f"Using catalog: {CATALOG}, schema: {SCHEMA}")


In [ ]:
from pyspark.sql.functions import regexp_replace

# Read distinct patient_ids from pseudo_incident_7d_labeled (created by notebook 05)
df_distinct_patient = (
    spark.table(f"{CATALOG}.{SCHEMA}.pseudo_incident_7d_labeled")
         .select("patient_id")
         .distinct()
)

# Derive device_id by replacing PSEUDO prefix with DEVICE
df_distinct_patient_device = df_distinct_patient.withColumn(
    "device_id", regexp_replace("patient_id", "PSEUDO", "DEVICE")
)

display(df_distinct_patient_device)


In [ ]:
# Save as patient_device table in the target catalog/schema
df_distinct_patient_device.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.{SCHEMA}.patient_device"
)
print(f"patient_device table written to {CATALOG}.{SCHEMA}.patient_device")
